# Fine-Tuning Model Penyakit Tomat dengan Dataset PlantDoc (Kondisi Nyata)

Notebook ini disiapkan untuk melatih ulang model klasifikasi penyakit daun tomat menggunakan **PlantDoc Dataset** dari Kaggle. Dataset ini sangat cocok karena berisi foto daun di **kondisi lapangan nyata (in-the-wild)**, bukan di laboratorium berlatar abu-abu.

Karena PlantDoc berisi berbagai macam tanaman (apel, jagung, kentang, dll.), notebook ini dilengkapi dengan **script penyaring otomatis** untuk hanya mengambil kelas daun tomat dan menyelaraskannya dengan model projek Anda.

## 1. Setup Kaggle API
1. Buka akun Kaggle Anda -> Settings -> Bagian **API Tokens**.
2. Klik **Generate New Token** dan salin token panjangnya (`KGAT_...`).
3. Jalankan cell di bawah ini, paste token tersebut, lalu tekan Enter.

In [ ]:
!pip install -q kaggle
import os
from getpass import getpass

token = getpass('Masukkan Kaggle API Token Anda: ')
os.environ['KAGGLE_API_TOKEN'] = token
print("\nKaggle API berhasil disetup!")

## 2. Download dan Ekstrak PlantDoc Dataset

In [ ]:
DATASET_NAME = "nirmalsankalana/plantdoc-dataset"

print("Mengunduh PlantDoc Dataset dari Kaggle...")
!kaggle datasets download -d $DATASET_NAME

import zipfile
zip_name = DATASET_NAME.split('/')[-1] + ".zip"
print("Mengekstrak dataset...")
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Ekstraksi selesai!")

## 3. Script Penyaring Otomatis (Filter Khusus Tomat)
Script di bawah ini akan secara otomatis memindai seluruh isi folder PlantDoc, mendeteksi folder penyakit tanaman tomat, lalu memindahkannya ke folder baru `dataset_tomato/` dengan nama kelas yang bersih sesuai dengan backend aplikasi Anda (`LABEL_MAP`).

In [ ]:
import os
import shutil

src_root = "dataset"
dest_root = "dataset_tomato"

# Pemetaan dari nama kelas PlantDoc ke LABEL_MAP backend Anda
target_mapping = {
    "Tomato bacterial spot leaf": "BACTERIAL SPOT",
    "Tomato leaf early blight": "EARLY BLIGHT",
    "Tomato leaf late blight": "LATE BLIGHT",
    "Tomato mold leaf": "LEAF MOLD",
    "Tomato Septoria leaf spot": "SEPTORIA LEAF SPOT",
    "Tomato Spider mites leaf": "SPIDER MITES",
    "Tomato target spot leaf": "TARGET SPOT",
    "Tomato leaf yellow virus": "YELLOW LEAF CURL VIRUS",
    "Tomato leaf mosaic virus": "MOSAIC VIRUS",
    "Tomato Leaf": "SEHAT"
}

def clean_name(name):
    return name.lower().replace(" ", "").replace("_", "").replace("-", "")

if os.path.exists(dest_root):
    shutil.rmtree(dest_root)
os.makedirs(dest_root, exist_ok=True)

subdirs = os.listdir(src_root)
splits = [d for d in subdirs if d.lower() in ['train', 'test', 'val', 'validation']]

if splits:
    for split in splits:
        os.makedirs(os.path.join(dest_root, split), exist_ok=True)
        split_path = os.path.join(src_root, split)
        for class_name in os.listdir(split_path):
            matched_key = None
            for key in target_mapping.keys():
                if clean_name(key) == clean_name(class_name):
                    matched_key = key
                    break
            if matched_key:
                src_class = os.path.join(split_path, class_name)
                dest_class = os.path.join(dest_root, split, target_mapping[matched_key])
                shutil.copytree(src_class, dest_class)
                print(f"[{split.upper()}] Berhasil menyaring: {class_name} -> {target_mapping[matched_key]}")
else:
    for class_name in subdirs:
        matched_key = None
        for key in target_mapping.keys():
            if clean_name(key) == clean_name(class_name):
                matched_key = key
                break
        if matched_key:
            src_class = os.path.join(src_root, class_name)
            dest_class = os.path.join(dest_root, target_mapping[matched_key])
            shutil.copytree(src_class, dest_class)
            print(f"Berhasil menyaring: {class_name} -> {target_mapping[matched_key]}")

print("\nSelesai! Gambar tomat berhasil dipisahkan ke folder 'dataset_tomato'.")

## 4. Persiapan Data (ImageDataGenerator)
Kita akan memuat gambar tomat hasil saringan dan mengurutkan index kelas secara eksplisit (0-9) agar cocok dengan backend.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

if os.path.exists("dataset_tomato/train"):
    DATA_DIR = "dataset_tomato/train"
    VAL_DIR = "dataset_tomato/test" if os.path.exists("dataset_tomato/test") else None
else:
    DATA_DIR = "dataset_tomato"
    VAL_DIR = None

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Urutan kelas ini WAJIB sama dengan array index di backend
CLASS_NAMES = [
    "BACTERIAL SPOT",
    "EARLY BLIGHT",
    "LATE BLIGHT",
    "LEAF MOLD",
    "SEPTORIA LEAF SPOT",
    "SPIDER MITES",
    "TARGET SPOT",
    "YELLOW LEAF CURL VIRUS",
    "MOSAIC VIRUS",
    "SEHAT"
]

# Buat folder kosong jika ada kelas yang tidak memiliki gambar di dataset baru
# untuk mencegah error Keras flow_from_directory
for class_name in CLASS_NAMES:
    os.makedirs(os.path.join(DATA_DIR, class_name), exist_ok=True)
    if VAL_DIR:
        os.makedirs(os.path.join(VAL_DIR, class_name), exist_ok=True)

if VAL_DIR:
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True
    )
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES
    )
    val_generator = val_datagen.flow_from_directory(
        VAL_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES
    )
else:
    datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        validation_split=0.2
)
    train_generator = datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        classes=CLASS_NAMES
    )
    val_generator = datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        classes=CLASS_NAMES
    )

print("\nMapping Index Kelas:")
for class_name, idx in train_generator.class_indices.items():
    print(f"Index {idx} -> {class_name}")

## 5. Membangun Model Transfer Learning (MobileNetV2)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Training Tahap 1 (Hanya Head)

In [ ]:
EPOCHS = 10
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)

## 7. Training Tahap 2 (Fine-Tuning)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), 
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_finetune = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

## 8. Simpan Model dan Download

In [ ]:
MODEL_NAME = "tomato_model.h5"
model.save(MODEL_NAME)
print(f"Model berhasil disimpan sebagai {MODEL_NAME}")

from google.colab import files
files.download(MODEL_NAME)